In [0]:
from pyspark.sql.functions import col, current_timestamp, regexp_extract, to_date

In [0]:
landing_path    = "/Volumes/skypoints/landing/redemptions"
checkpoint_path = "/Volumes/skypoints/bronze/redemptions/redemption_checkpoints"

In [0]:
# Every field is STRING on purpose (bronze = raw, no casting).
# The nested array is kept as ARRAY<STRUCT<...>> with STRING fields inside.
# Anything NOT listed here (memberId, source_system, fee_usd, channel, route ...)
# is not dropped: Auto Loader puts it into _rescued_data.
file_schema = """
    member_id    STRING,
    feed_date    STRING,
    redemptions  ARRAY<STRUCT<
        txn_id          : STRING,
        txn_date        : STRING,
        partner         : STRING,
        miles_redeemed  : STRING,
        status          : STRING
    >>
"""

In [0]:
%sql
CREATE TABLE IF NOT EXISTS skypoints.bronze.raw_redemptions
(
    -- fields from the redemption file, all raw text
    member_id          STRING,
    feed_date          STRING,
    redemptions        ARRAY<STRUCT<
                           txn_id         : STRING,
                           txn_date       : STRING,
                           partner        : STRING,
                           miles_redeemed : STRING,
                           status         : STRING
                       >> COMMENT 'Nested array kept as-is; every inner field is text. Explode in silver',

    -- audit
    _rescued_data      STRING    COMMENT 'Fields not in the schema: memberId, source_system, fee_usd, channel, route ...',
    source_file_name   STRING    COMMENT 'e.g. SKYPOINTS_REDEMPTIONS_20240106_04544984.json',
    file_date          DATE      COMMENT 'Parsed from the file name',
    ingestion_timestamp TIMESTAMP COMMENT 'When the row was loaded into bronze'
)
USING DELTA
COMMENT 'Bronze: raw redemption rows (one row per member per feed, nested array), append-only, no casting';

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")          # newline-delimited JSON, one member per line
    .option("rescuedDataColumn", "_rescued_data")
    .schema(file_schema)
    .load(landing_path)
    .withColumn("source_file_name", col("_metadata.file_name"))
    .withColumn("file_date", to_date(regexp_extract(col("_metadata.file_name"), r"_(\d{8})_", 1), "yyyyMMdd"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .select(
        "member_id", "feed_date", "redemptions",
        "_rescued_data", "source_file_name", "file_date", "ingestion_timestamp"
    )
)

In [0]:
(
    df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)                   # process what is there, then stop
    .toTable("skypoints.bronze.raw_redemptions")
)

In [0]:
%sql
select * from skypoints.bronze.raw_redemptions;